Enable GPU T4 and Internet. Attach the updated courtroom-kaggle Dataset; enable NVIDIA_API_KEY Secret. Frozen BGE-base-en-v1.5 BERT embeddings run on CUDA; first use downloads about440MB. Actor observations use their cosine relevance. Step penalty .01 and output-token penalty .00001; call cost0. Save outputs/checkpoints before ending the session.

In [ ]:
import os, sys, shutil, subprocess, zipfile
from pathlib import Path
from kaggle_secrets import UserSecretsClient

PROJECT = Path('/kaggle/working/cocolofa_pard')
archives = list(Path('/kaggle/input').rglob('courtroom-kaggle.zip'))
if not PROJECT.exists():
    if len(archives) == 1:
        with zipfile.ZipFile(archives[0]) as z:
            z.extractall('/kaggle/working')
    else:
        candidates = list(Path('/kaggle/input').rglob('courtroom_rl.kaggle.yaml'))
        assert len(candidates) == 1, 'Attach the courtroom-kaggle Dataset first'
        shutil.copytree(candidates[0].parent.parent, PROJECT)
os.chdir(PROJECT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-kaggle.txt'], check=True)
os.environ['NVIDIA_API_KEY'] = UserSecretsClient().get_secret('NVIDIA_API_KEY')
os.environ['NVIDIA_BASE_URL'] = 'https://integrate.api.nvidia.com/v1'
os.environ['NVIDIA_MODEL'] = 'meta/llama-3.2-11b-vision-instruct'
os.environ['PYTHONUNBUFFERED'] = '1'
import torch
assert torch.cuda.is_available(), 'Enable GPU T4 in Notebook Settings'
print('PPO device:', torch.cuda.get_device_name(0), '| LLM: NVIDIA API')


In [ ]:
# Real frozen BGE + V4 CUDA rollout/update/checkpoint, no API calls.
subprocess.run([sys.executable, '-u', 'scripts/check_kaggle_gpu.py', '--config', 'configs/courtroom_rl.v4.kaggle.yaml'], check=True)


In [ ]:
# First run: None. To resume, set the path to YOUR Kaggle last.pt.
# Attach saved output as a Dataset if resuming in a new session.
RESUME = None
# Example: RESUME = '/kaggle/input/my-training-output/cocolofa_pard/runs/courtroom-v4-.../last.pt'
cmd = [sys.executable, '-u', '-m', 'src.courtroom_rl.runner', 'train',
       '--config', 'configs/courtroom_rl.v4.kaggle.yaml',
       '--epochs', '1', '--batch-size', '8', '--workers', '4', '--dev-limit', '66',
       '--dev-every', '10', '--warmup-episodes', '8', '--seed', '0', '--log-steps']
if RESUME:
    # Copy the run directory so inherited best.pt and its provenance travel too.
    source = Path(RESUME).resolve()
    destination = PROJECT / 'resume_inputs' / source.parent.name
    if not destination.exists():
        shutil.copytree(source.parent, destination)
    cmd += ['--resume', str(destination / source.name)]
subprocess.run(cmd, check=True)


In [ ]:
# Run after training or interrupting: create a small downloadable recovery bundle.
# Includes all run metadata, last.pt/best.pt, logs and assessor cache, no API key.
import tarfile
artifact = Path('/kaggle/working/courtroom-training-output.tar.gz')
with tarfile.open(artifact, 'w:gz') as tar:
    for relative in ['runs', 'cache/courtroom_rl/assessor.joblib']:
        path = PROJECT / relative
        if path.exists():
            tar.add(path, arcname='cocolofa_pard/' + relative)
print('Saved:', artifact)
